# Assignment 3 — Two Averages, One Argument
**Sahibzadi Minal Ghori | Rep 37170**

This notebook rebuilds the cleaned dataset directly from the supplied original CSV. It does not use the earlier unrelated 2025/8-item file. The supplied data label identifies the observations as illustrative teaching data, not actual PBS observations.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Keep the original CSV in the same folder as this notebook.
DATA_FILE = Path('teaching_prices_messy_original.csv')
raw = pd.read_csv(DATA_FILE)
print('Original shape:', raw.shape)
print('Columns:', raw.columns.tolist())
raw.head()

Original shape: (97, 6)
Columns: ['date', 'item', 'city', 'price', 'unit', 'data_label']


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


## 1. Clean the original file transparently

Rules used: (1) remove only exact duplicate rows; (2) standardize item-name capitalization; (3) remove the thousands separator from `1,250` so it parses as 1250; (4) retain missing prices as missing rather than imputing; (5) retain unusual values and flag them rather than silently changing or deleting them.

In [2]:
df = raw.copy()
duplicate_count = int(df.duplicated().sum())
df = df.drop_duplicates().copy()
df['item'] = df['item'].astype(str).str.strip().str.replace(r'\s+', ' ', regex=True).str.title()
df['price_pkr'] = pd.to_numeric(df['price'].astype(str).str.replace(',', '', regex=False), errors='coerce')
df['outlier_flag'] = df['price_pkr'].eq(9999)
df['review_flag'] = df['price_pkr'].isin([1250, 108.5])
df['cleaning_note'] = 'No change; numeric value retained'
df.loc[df['price_pkr'].isna(), 'cleaning_note'] = 'Missing price retained; no imputation'
df.loc[df['price_pkr'].eq(1250), 'cleaning_note'] = 'Comma removed; retained as 1250 and flagged for source verification'
df.loc[df['price_pkr'].eq(9999), 'cleaning_note'] = 'Extreme value retained and flagged for sensitivity analysis'
df.loc[df['price_pkr'].eq(108.5), 'cleaning_note'] = 'Unusual relative to adjacent Oil B City 1 values; retained for verification'
clean = df[['date','item','city','price_pkr','unit','data_label','outlier_flag','review_flag','cleaning_note']].copy()
clean.to_csv('teaching_prices_clean_corrected.csv', index=False)
print('Exact duplicates removed:', duplicate_count)
print('Rows after deduplication:', len(clean))
print('Missing prices:', clean['price_pkr'].isna().sum())
print('Flagged 9,999 values:', clean['outlier_flag'].sum())
clean.head()

Exact duplicates removed: 1
Rows after deduplication: 96
Missing prices: 1
Flagged 9,999 values: 1


,date,item,city,price_pkr,unit,data_label,outlier_flag,review_flag,cleaning_note
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,False,False,No change; numeric value retained
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,False,False,No change; numeric value retained
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,False,False,No change; numeric value retained
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,False,False,No change; numeric value retained
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,False,False,No change; numeric value retained


## 2. Mean, median, and first-to-last percentage change by item

Mean and median use all available numeric observations for each item, including flagged values. The first-to-last percentage change compares the average of available city observations on the earliest and latest dates for each item. If one city value is missing on an endpoint date, the available value is used and this limitation is noted.

In [3]:
valid = clean.dropna(subset=['price_pkr']).copy()
item_stats = valid.groupby('item')['price_pkr'].agg(
    observations='count', mean='mean', median='median'
).reset_index()

change_rows = []
for item, group in clean.groupby('item'):
    daily = group.groupby('date')['price_pkr'].mean().dropna()
    first_date, last_date = daily.index[0], daily.index[-1]
    first_value, last_value = float(daily.iloc[0]), float(daily.iloc[-1])
    pct_change = (last_value - first_value) / first_value * 100
    change_rows.append({
        'item': item, 'first_date': first_date, 'first_available_city_average': first_value,
        'last_date': last_date, 'last_available_city_average': last_value,
        'first_to_last_percent_change': pct_change
    })
change_table = pd.DataFrame(change_rows)
print('Mean and median:')
display(item_stats.round(2))
print('First-to-last change:')
display(change_table.round(2))

Mean and median:


,item,observations,mean,median
0,Oil B,25,316.18,324.00
1,Pulse C,23,205.91,206.00
2,Staple A,23,160.89,112.50
3,Sweetener D,24,576.81,167.75


First-to-last change:


,item,first_date,first_available_city_average,last_date,last_available_city_average,first_to_last_percent_change
0,Oil B,2026-01-01,301.0,2026-03-19,347.0,15.28
1,Pulse C,2026-01-01,200.0,2026-03-19,210.5,5.25
2,Staple A,2026-01-01,104.0,2026-03-19,117.5,12.98
3,Sweetener D,2026-01-01,152.0,2026-03-19,181.5,19.41


## 3. One item with and without the flagged value

The extreme Sweetener D observation is 9,999. The comparison below removes only that observation for the sensitivity calculation; it does not delete it from the cleaned dataset. This illustrates how an extreme value can pull the mean while the median is more resistant.

In [4]:
sweetener = valid.loc[valid['item'].eq('Sweetener D'), 'price_pkr']
sweetener_without_flag = sweetener.loc[sweetener.ne(9999)]
sensitivity = pd.DataFrame([
    {'case': 'With flagged value', 'n': len(sweetener), 'mean': sweetener.mean(), 'median': sweetener.median()},
    {'case': 'Without 9,999 (sensitivity check)', 'n': len(sweetener_without_flag), 'mean': sweetener_without_flag.mean(), 'median': sweetener_without_flag.median()}
])
display(sensitivity.round(2))

,case,n,mean,median
0,With flagged value,24,576.81,167.75
1,"Without 9,999 (sensitivity check)",23,167.15,167.50


## 4. Data-quality checks and limitations

In [5]:
print('Original rows:', len(raw))
print('Exact duplicate rows:', raw.duplicated().sum())
print('Cleaned rows:', len(clean))
print('Missing numeric prices:', clean['price_pkr'].isna().sum())
print('Outlier flag count:', clean['outlier_flag'].sum())
print('Other review flags (1,250 and 108.5):', clean['review_flag'].sum())
print('Date range:', clean['date'].min(), 'to', clean['date'].max())
print('Items:', sorted(clean['item'].unique()))
print('Cities:', sorted(clean['city'].unique()))
print('Data label:', clean['data_label'].dropna().unique())

Original rows: 97
Exact duplicate rows: 1
Cleaned rows: 96
Missing numeric prices: 1
Outlier flag count: 1
Other review flags (1,250 and 108.5): 2
Date range: 2026-01-01 to 2026-03-19
Items: ['Oil B', 'Pulse C', 'Staple A', 'Sweetener D']
Cities: ['City 1', 'City 2']
Data label: ['ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OBSERVATIONS']


### Interpretation

The mean is sensitive to extreme values because it uses every value in the sum. The median is the middle observation after sorting and is more resistant to a single extreme value. A large difference between mean and median is a warning to inspect the distribution and source records; it is not proof that an observation is wrong.

The original data explicitly labels itself illustrative. These calculations therefore describe only this teaching dataset. They cannot establish actual Pakistan-wide prices or causes of price changes. The file also does not include a separate Week 2 cleaned dataset; the sensitivity analysis above uses the supplied original as a clearly stated substitute.